# Automatic Differentiation: Hands-On with Clad

**Clad** is a source-to-source automatic-differentiation (AD) plugin for the Clang/LLVM
compiler. Given a C++ function, it *generates the C++ source of its derivative* at
compile time (here, at Cling JIT time) which is then compiled like any other code. The
derivative is therefore **exact** (not finite differences) and as fast as hand-written
code.

This notebook is a hands-on tour in five sections:

1. **Clad fundamentals** — forward vs. reverse mode, inspecting the generated code, driving Clad from Python, and Hessians.
2. **When branches lie to the differentiator** — discontinuities and the traps they set for AD.
3. **Jacobians, jvp and vjp** — vector-valued outputs, and the forward/reverse duality that maps onto JAX.
4. **Differentiable detector design** — putting a gradient to work in an optimizer.
5. **Measuring the Z mass from real CMS open data** — a binned likelihood fit, with gradient *and* Hessian from one function.

We write the C++ with ROOT's `%%cpp` cell magic: `%%cpp --declare` to *define* functions
and classes, and plain `%%cpp` to run the `clad::*` call that *triggers code generation*.
The generated function then appears as a normal `ROOT.<name>`, which we call from Python.

## Setup

Importing `ROOT` inside a Jupyter kernel automatically registers the `%%cpp` magic. One
helper, `show_generated()`, captures the C++ source that Clad's `.dump()` prints (it goes
to the process's stdout, which we redirect so it shows up in the cell).

**Naming of the generated functions.** Clad names them predictably from the request:
reverse mode is `<f>_grad`, the Hessian is `<f>_hessian`, and forward mode is `<f>_darg<k>`
(one function per input `k`). The one wrinkle: when the differentiated argument is an
**array** *and* the function takes further arguments — as our fit likelihoods do, passing
the parameter array **and** a `const double*` of data — Clad appends an index,
`<f>_grad_0` / `<f>_hessian_0`. Plain-scalar functions (even `f(x, y)` with several
arguments), and a lone array argument, keep the bare name. We call each generated function
by its exact name below — no guessing.

> Note: `%%cpp --declare` goes through Cling, which does **not** allow redefining a symbol.
> Re-running a declaration cell (or re-running the notebook) needs a kernel restart first.

In [ ]:
import os, sys, tempfile
import numpy as np
import ROOT
from iminuit import Minuit
import matplotlib.pyplot as plt
%matplotlib inline

def show_generated(stmt):
    "Run a clad codegen+dump statement and return the C++ source it prints."
    sys.stdout.flush()
    saved = os.dup(1)
    with tempfile.TemporaryFile(mode="w+") as tmp:
        os.dup2(tmp.fileno(), 1)
        try:
            ROOT.gInterpreter.ProcessLine(stmt)
            ROOT.gInterpreter.ProcessLine("std::cout.flush(); fflush(stdout);")
        finally:
            os.dup2(saved, 1); os.close(saved)
        tmp.seek(0)
        return tmp.read()

# Warm up the Cling + Clad JIT now (a few seconds), so the first exercise cell is snappy.
ROOT.gInterpreter.Declare("#include <Math/CladDerivator.h>")
ROOT.gInterpreter.Declare("double _warm(double z){ return z*z; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(_warm);")
print("ROOT", ROOT.gROOT.GetVersion())

## 1. Clad fundamentals

Three entry points, each taking a function and returning a callable:

| call | mode | gives |
|------|------|-------|
| `clad::differentiate(f, "x")` | forward | one directional derivative |
| `clad::gradient(f)` | reverse | all first partials at once |
| `clad::hessian(f, "x,y")` | second order | the full Hessian |

Our running example is $f(x, y) = x^2 y + \sin x$, with
$\partial f/\partial x = 2xy + \cos x$, $\partial f/\partial y = x^2$, and
$\partial^2 f/\partial x^2 = 2y - \sin x$, $\partial^2 f/\partial x\,\partial y = 2x$,
$\partial^2 f/\partial y^2 = 0$.

In [ ]:
%%cpp --declare
#include <Math/CladDerivator.h>
#include <cmath>
double f(double x, double y) { return x*x*y + std::sin(x); }

In [ ]:
x, y = 1.3, 2.0

### 1.1 Forward mode — `clad::differentiate`

Forward mode propagates one input's perturbation through the computation, so **one call
gives one partial derivative**; a separate generated function is produced per input. Its
cost scales with the number of **inputs** — good for functions with few inputs, many outputs.

In [ ]:
%%cpp
clad::differentiate(f, "x");   // -> f_darg0
clad::differentiate(f, "y");   // -> f_darg1

In [ ]:
dfdx = ROOT.f_darg0(x, y)   # forward mode RETURNS the derivative directly
dfdy = ROOT.f_darg1(x, y)
print(f"df/dx = {dfdx:.6f}   (analytic {2*x*y + np.cos(x):.6f})")
print(f"df/dy = {dfdy:.6f}   (analytic {x*x:.6f})")

### 1.2 Reverse mode — `clad::gradient`

Reverse mode propagates the output's sensitivity backwards, so **one call gives all
partials together**. Its cost scales with the number of **outputs** (one sweep) — the
mode you want for fitting/ML, where one scalar loss depends on many parameters. For
scalar arguments the gradient comes back through trailing output references.

In [ ]:
%%cpp
clad::gradient(f);   // -> f_grad

In [ ]:
# scalar outputs come back through trailing double* args; use length-1 numpy arrays
# for them -- the same array convention as the multi-parameter case below
gx, gy = np.zeros(1), np.zeros(1)
ROOT.f_grad(x, y, gx, gy)
print(f"grad f = ({gx[0]:.6f}, {gy[0]:.6f})   "
      f"(analytic ({2*x*y + np.cos(x):.6f}, {x*x:.6f}))")

### 1.3 Inspecting the generated code — `.dump()`

Clad's entry points return an object whose `.dump()` prints the generated C++ source. The
printed source is plain C++ — you can paste it into a codebase that does not link Clad at
all. (`.dump()` writes to the process stdout via an LLVM stream, so here we run it through
the `show_generated()` helper to capture it into the cell.)

In [ ]:
print(show_generated('{ auto d = clad::differentiate(f, "x"); d.dump(); }'))

In [ ]:
print(show_generated('{ auto g = clad::gradient(f); g.dump(); }'))

### 1.4 From Python: array parameters

Real models pack parameters into a `double*` and differentiate w.r.t. the whole array;
from Python you pass a numpy array for the parameters and a numpy array to receive the
gradient. Two rules that bite people: pass **contiguous float64** arrays, and **zero the
output array** before each call — Clad *accumulates* into it.

In [ ]:
%%cpp --declare
#include <Math/CladDerivator.h>
#include <cmath>
double g(double *p) { return p[0]*p[0]*p[1] + std::sin(p[0]); }   // same f, packed

In [ ]:
%%cpp
clad::gradient(g, "p");   // -> g_grad

In [ ]:
p = np.array([x, y])
grad = np.zeros(2)                 # must be zeroed: Clad adds into it
ROOT.g_grad(p, grad)              # lone array argument -> bare name, no _0
print("params p  =", p)
print("grad g(p) =", grad, "  (same numbers, array-valued)")

### 1.5 Hessians — `clad::hessian`

`clad::hessian` gives the matrix of second derivatives (it composes forward over reverse
mode internally) as a flattened $n\times n$ buffer; reshape to $(n, n)$. Its headline use
is uncertainties: for a negative log-likelihood, the parameter covariance is the inverse
Hessian at the minimum, $C = H^{-1}$.

In [ ]:
%%cpp
clad::hessian(f, "x,y");   // -> f_hessian

In [ ]:
H = np.zeros(4)
ROOT.f_hessian(x, y, H)
H = H.reshape(2, 2)
print("Hessian =")
print(H)
print("analytic = [[2y - sin(x), 2x], [2x, 0]] =",
      [[round(2*y - np.sin(x), 4), 2*x], [2*x, 0.0]])

**Exercise.** The gentlest possible Clad task. The function `cube(x) = x*x*x` has derivative $3x^2$, so
its derivative at $x = 2$ is $12$. Use Clad to generate the derivative and check it.

*(These exercise cells drive Clad from Python via `ROOT.gInterpreter` so each fits in a
single cell — it is exactly what the `%%cpp` magic does under the hood.)*

In [ ]:
# EXERCISE: differentiate cube(x) = x*x*x with Clad and evaluate the derivative at x = 2.
#   1. declare it:      ROOT.gInterpreter.Declare("double cube(double x){ return x*x*x; }")
#   2. differentiate:   ROOT.gInterpreter.ProcessLine("clad::gradient(cube);")
#   3. call cube_grad(2.0, out) with a length-1 numpy array 'out', then print out[0]
# (Write your version here; running this stub as-is does nothing.)


**Solution.**

In [ ]:
ROOT.gInterpreter.Declare("double cube(double x){ return x*x*x; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(cube);")
d = np.zeros(1)
ROOT.cube_grad(2.0, d)
print("cube'(2) =", d[0], " (analytic 12)")


## 2. When branches lie to the differentiator

Everywhere else in this notebook AD "just works". Here we show the sharp edges. The
one-sentence lesson:

> Automatic differentiation differentiates the **code path you actually execute** — which
> is not always the **math you meant**.

This is different from a bug in the AD tool itself (those exist too). Here Clad is
faithful: it returns the exact derivative of the branch that ran. The trouble is that an
innocent-looking `if` can make that branch's derivative disagree with the function you
thought you wrote — and the function *value* can look perfectly fine while the derivative
is silently corrupted.

In [ ]:
%%cpp --declare
#include <Math/CladDerivator.h>
#include <cmath>
// f(x) = x*x - 3x = x(x-3): a parabola through the origin, so f(0) = 0.
double para_plain(double x) { return x*x - 3.0*x; }
double para_opt(double x) {
    if (x == 0.0) return 0.0;            // "optimization": we KNOW f(0) = 0
    return x*x - 3.0*x;
}
// a "stabilizing guard" that snaps small |x| to zero
double para_guard(double x) {
    if (std::fabs(x) < 0.5) return 0.0;
    return x*x - 3.0*x;
}
// a kink (ReLU), two spellings differing only in < vs <=
double relu_lt(double x) { if (x <  0.0) return 0.0; return x; }
double relu_le(double x) { if (x <= 0.0) return 0.0; return x; }
// smooth stand-in: softplus'(x) = sigmoid(x)
double softplus(double x) { return std::log1p(std::exp(x)); }

In [ ]:
%%cpp
clad::gradient(para_plain);
clad::gradient(para_opt);
clad::gradient(para_guard);
clad::gradient(relu_lt);
clad::gradient(relu_le);
clad::gradient(softplus);

In [ ]:
def clad_grad(name, xv):
    # every function here takes a single scalar -> the generated name is just <name>_grad
    d = np.zeros(1); getattr(ROOT, name + "_grad")(xv, d); return d[0]
def fin_diff(name, xv, h=1e-6):
    fnc = getattr(ROOT, name); return (fnc(xv + h) - fnc(xv - h)) / (2*h)
print("helpers ready")

### 2.1 The fast path: correct value, silently wrong derivative

$f(x) = x(x-3)$ has $f(0) = 0$, so a shortcut `if (x == 0) return 0;` looks harmless. The
value is right everywhere and the derivative matches the honest version at *every* point
except $x = 0$ — where AD reports $0$ instead of the true $f'(0) = -3$. A unit test at any
random $x$ passes; the one point you special-cased is the one AD gets wrong.

In [ ]:
print(f"{'x':>6} {'value':>10} {'plain f_x':>11} {'opt f_x':>10} {'finite diff':>12}")
for xv in [-1.0, 0.0, 1.0, 2.0]:
    print(f"{xv:6.1f} {ROOT.para_opt(xv):10.3f} {clad_grad('para_plain', xv):11.3f} "
          f"{clad_grad('para_opt', xv):10.3f} {fin_diff('para_opt', xv):12.3f}")

### 2.2 A kink: AD hands you one side, and `<` vs `<=` picks which

At a kink the derivative does not exist. AD returns a one-sided value, and moving equality
to the other branch (`<` vs `<=`) flips the reported slope between 1 and 0 — a coin toss
your optimizer would silently inherit. A central finite difference reports yet a third
answer (0.5, the average of the two sides).

In [ ]:
print(f"{'x':>6} {'relu_lt f_x':>13} {'relu_le f_x':>13} {'finite diff':>13}")
for xv in [-1.0, 0.0, 1.0]:
    print(f"{xv:6.1f} {clad_grad('relu_lt', xv):13.3f} "
          f"{clad_grad('relu_le', xv):13.3f} {fin_diff('relu_lt', xv):13.3f}")

### 2.3 A stabilizing guard: the same disease over a region

A snap-to-zero guard corrupts value *and* slope across a whole band. Worse, a finite-diff
check that stays inside the band is fooled too (it also sees zero).

In [ ]:
print(f"{'x':>6} {'true value':>12} {'guard value':>12} {'guard f_x':>11} {'finite diff':>12}")
for xv in [-1.0, -0.3, 0.3, 1.0]:
    print(f"{xv:6.1f} {ROOT.para_plain(xv):12.3f} {ROOT.para_guard(xv):12.3f} "
          f"{clad_grad('para_guard', xv):11.3f} {fin_diff('para_guard', xv):12.3f}")

### 2.4 Detect, and fix

A cheap habit that pays off: spot-check every generated gradient against a central finite
difference with a not-too-small step. It flags the point trap and the kink immediately
(region guards can still hide, so also just *know* where your branches are).

The real fix is to remove the discontinuity, not to differentiate around it: drop the
needless fast path (`para_plain` is already exact), replace a kink with a smooth curve
(softplus for ReLU), replace a hard cut with a sigmoid (the detector-acceptance fix in
section 4), and keep parameters out of branch conditions (evaluating a binned model at
fixed bin centers, as in section 5).

In [ ]:
print(f"{'function':12} {'x':>5} {'clad':>9} {'finite diff':>12}  verdict")
for name, xv in [("para_opt",0.0), ("relu_le",0.0), ("para_guard",0.0), ("para_plain",0.0)]:
    c = clad_grad(name, xv); fdv = fin_diff(name, xv, h=1e-2)   # wide step
    bad = abs(c - fdv) > 1e-2 * max(1.0, abs(fdv))
    print(f"{name:12} {xv:5.1f} {c:9.3f} {fdv:12.3f}  "
          f"{'MISMATCH -> inspect' if bad else 'ok'}")
print()
print(f"softplus'(0) = {clad_grad('softplus', 0.0):.3f}  (a smooth 0.5, vs ReLU's 0-or-1)")

**Exercise.** In section 2.1, `para_opt` returns the right *value* but Clad reports a derivative of $0$
at $x = 0$ instead of the true $-3$, because of the `if (x == 0) return 0;` fast path. Write
a fixed version with the same values but a correct derivative everywhere, and check that its
gradient at $x = 0$ is $-3$.

In [ ]:
# EXERCISE: write 'para_fixed' with the same values as para_opt but a correct derivative,
# differentiate it, and verify para_fixed'(0) == -3.
#   Hint: f(0) = 0 already falls out of the plain formula x*x - 3*x -- the special case
#   only adds a discontinuity in the derivative, so just drop it.
#   You can reuse the clad_grad(name, x) helper defined earlier in section 2.


**Solution.**

In [ ]:
ROOT.gInterpreter.Declare("double para_fixed(double x){ return x*x - 3.0*x; }")
ROOT.gInterpreter.ProcessLine("clad::gradient(para_fixed);")
print("para_fixed(0)  =", ROOT.para_fixed(0.0), " (same value as para_opt)")
print("para_fixed'(0) =", clad_grad("para_fixed", 0.0), " (fixed: -3, not 0)")


## 3. Jacobians, jvp and vjp

The tutorial gave forward (`differentiate`) and reverse (`gradient`) on a **scalar**
output. When the output becomes a **vector**, the gradient becomes a full **Jacobian**,
and the two dual ways of touching it get the names the JAX talks use:

- forward mode $\to$ **jvp** ("push a tangent forward", $J\,v$)
- reverse mode $\to$ **vjp** ("pull a cotangent back", $J^\top w$)

The physics: a track is measured as $(p_T, \eta, \phi)$ with a covariance matrix, and we
need the momentum in Cartesian coordinates,
$$p_x = p_T\cos\phi, \qquad p_y = p_T\sin\phi, \qquad p_z = p_T\sinh\eta,$$
a map $\mathbb{R}^3 \to \mathbb{R}^3$. To move the covariance we need its Jacobian $J$ and
then $C' = J\,C\,J^\top$.

| Clad | gives | JAX |
|------|-------|-----|
| `clad::jacobian(f)` | full $J$, **forward** | `jax.jacfwd` |
| `clad::differentiate(f, "x_k")` | column $k = J e_k$ | `jax.jvp` |
| `clad::gradient(f_component_j)` | row $j = e_j^\top J$ | `jax.vjp` |
| (stack all component gradients) | reverse-mode $J$ | `jax.jacrev` |

**`clad::jacobian` uses vectorized *forward* mode** — it seeds an identity matrix of input
tangents and pushes them all through in one sweep, so it is the `jacfwd` analogue; the
reverse-mode Jacobian (`jacrev`) is the stack of per-output gradients in section 3.3.
(Clad's own docs are inconsistent on this point, so we do not take it on faith: the
`.dump()` in section 3.1 shows the generated code is built from `*_pushforward` calls, and
in the source `JacobianModeVisitor` derives `VectorPushForwardModeVisitor`.)

One convention detail below: an **output** array parameter must carry a `_clad_out_` prefix
so Clad treats it as the output rather than another independent variable — otherwise it
pads the Jacobian with extra columns. With the prefix, the derivative matrix is a clean
$n_\text{out}\times n_\text{in}$.

In [ ]:
%%cpp --declare
#include <Math/CladDerivator.h>
#include <cmath>
// Vector form, for clad::jacobian. The output array carries the _clad_out_ prefix so Clad
// treats it as the OUTPUT (not another independent):  in = (pT, eta, phi) -> (px, py, pz).
void trk(double *in, double *_clad_out_out) {
    double pT = in[0], eta = in[1], phi = in[2];
    _clad_out_out[0] = pT * std::cos(phi);
    _clad_out_out[1] = pT * std::sin(phi);
    _clad_out_out[2] = pT * std::sinh(eta);
}
// scalar component form, for the jvp (differentiate) and vjp (gradient) views
double px(double pT, double eta, double phi) { return pT * std::cos(phi); }
double py(double pT, double eta, double phi) { return pT * std::sin(phi); }
double pz(double pT, double eta, double phi) { return pT * std::sinh(eta); }

// clad::jacobian shim. clad::jacobian returns a functor whose .execute takes the inputs,
// the outputs, and one clad::matrix<double>* per array parameter (the caller allocates
// them; clad::matrix comes in with <Math/CladDerivator.h>, no extra include). Thanks to
// the _clad_out_ prefix the derivative matrix d_out is a clean 3 x 3 (rows = outputs,
// cols = inputs). Generate + call in one transaction, then copy the matrix out flat.
void trk_jacobian(double *in, double *Jflat) {
    auto jac = clad::jacobian(trk, "in");
    double out[3];
    clad::matrix<double> d_in(3, 3), d_out(3, 3);
    jac.execute(in, out, &d_in, &d_out);
    for (int i = 0; i < 3; ++i)
        for (int k = 0; k < 3; ++k)
            Jflat[i * 3 + k] = d_out[i][k];
}

In [ ]:
%%cpp
// reverse-mode rows (vjp) and forward-mode columns (jvp) for each component
clad::gradient(px);
clad::gradient(py);
clad::gradient(pz);
clad::differentiate(px, "pT");  clad::differentiate(px, "eta");  clad::differentiate(px, "phi");
clad::differentiate(py, "pT");  clad::differentiate(py, "eta");  clad::differentiate(py, "phi");
clad::differentiate(pz, "pT");  clad::differentiate(pz, "eta");  clad::differentiate(pz, "phi");

In [ ]:
COMP = ["px", "py", "pz"]        # output index j
VARS = ["pT", "eta", "phi"]      # input index k
pT, eta, phi = 10.0, 0.5, 0.3
xtrk = np.array([pT, eta, phi])
analytic = np.array([[np.cos(phi),  0.0,              -pT*np.sin(phi)],
                     [np.sin(phi),  0.0,               pT*np.cos(phi)],
                     [np.sinh(eta), pT*np.cosh(eta),   0.0]])
print("evaluation point (pT, eta, phi) =", xtrk)

### 3.1 The full Jacobian in one call — `clad::jacobian`

In [ ]:
J = np.zeros(9)
ROOT.trk_jacobian(xtrk, J)
J = J.reshape(3, 3)
print("J = d(px,py,pz)/d(pT,eta,phi) =")
print(np.array2string(J, precision=5))
assert np.allclose(J, analytic), "clad::jacobian disagrees with the analytic Jacobian"
print("matches analytic: True")

The generated code settles the forward-vs-reverse question directly: it is built from
`*_pushforward` calls (vectorized forward mode), and `indepVarCount = _d_vector_in->rows()`
(3, not 6) shows the `_clad_out_` prefix kept the derivative matrix a clean 3 x 3.

In [ ]:
print(show_generated('{ auto j = clad::jacobian(trk, "in"); j.dump(); }'))

### 3.2 Forward mode = jvp: build the columns

`clad::differentiate(f, "x_k")` propagates a unit perturbation of input $k$ — a jvp against
the tangent $e_k$ — and yields column $k$ of $J$. A general $J\,v$ is the linear combination
$\sum_k v_k\,(\text{column } k)$, exactly what `jax.jvp` returns. `clad::jacobian` above *is*
this, vectorized: all columns pushed in one sweep. Cost scales with the number of inputs —
use when $J$ is tall.

In [ ]:
Jcol = np.empty((3, 3))
for j, c in enumerate(COMP):
    for k in range(3):
        Jcol[j, k] = getattr(ROOT, f"{c}_darg{k}")(pT, eta, phi)   # e.g. ROOT.px_darg0
print("columns from forward mode match clad::jacobian:", np.allclose(Jcol, J))

# a general "push a tangent forward": J.v for a 1-sigma-ish input shift v
v = np.array([0.15, 0.002, 0.0015])
jvp = J @ v
eps = 1e-6
out_p, out_m = np.zeros(3), np.zeros(3)
ROOT.trk(xtrk + eps*v, out_p)
ROOT.trk(xtrk - eps*v, out_m)
print("push tangent v =", v)
print("  J.v (forward)      =", np.array2string(jvp, precision=6))
print("  finite-diff of trk =", np.array2string((out_p - out_m)/(2*eps), precision=6))

### 3.3 Reverse mode = vjp: build the rows

`clad::gradient` of scalar component $j$ is the gradient of output $j$ w.r.t. all inputs =
row $j$ of $J$ — a vjp against the cotangent $e_j$. The scalar `clad::gradient` from the
tutorial was already exactly this with a single output (a vjp with cotangent $w = 1$), and
so is every fit still to come (detector loss, Z-peak NLL). Stacking these rows is the
reverse-mode Jacobian (`jax.jacrev`). Cost scales with the number of outputs — use when $J$
is wide; a scalar output is the extreme.

In [ ]:
Jrow = np.empty((3, 3))
for j, c in enumerate(COMP):
    dpT, deta, dphi = np.zeros(1), np.zeros(1), np.zeros(1)   # length-1 double* out-args
    getattr(ROOT, f"{c}_grad")(pT, eta, phi, dpT, deta, dphi)   # e.g. ROOT.px_grad
    Jrow[j] = [dpT[0], deta[0], dphi[0]]
print("rows from reverse mode match clad::jacobian:", np.allclose(Jrow, J))

w = np.array([1.0, 0.0, 0.0])   # cotangent selecting px
print("pull cotangent w =", w, " -> w^T J =", np.array2string(w @ J, precision=6), "(= row px)")

# hard cross-check: a broken shim would otherwise flow a wrong J into C' = J C J^T
assert np.allclose(J, Jcol) and np.allclose(J, Jrow), "the three Jacobian builds disagree"
print("clad::jacobian, forward columns, and reverse rows all agree: True")

### 3.4 The payoff: propagate the covariance, $C' = J\,C\,J^\top$

This is the matrix generalization of the scalar error-propagation formula
$\sigma^2 = g^\top C\,g$, where the gradient $g$ is a one-row Jacobian — a vjp with cotangent
$1$. We check the propagated covariance is symmetric and positive-definite.

In [ ]:
sigma = np.array([0.15, 0.0020, 0.0015])          # sigma_pT [GeV], sigma_eta, sigma_phi
corr = np.array([[1.0,  0.10, -0.20],
                 [0.10, 1.0,   0.05],
                 [-0.20, 0.05, 1.0]])
C_in = np.outer(sigma, sigma) * corr
Cp = J @ C_in @ J.T
evals = np.linalg.eigvalsh(Cp)
print("C' (px,py,pz) [GeV^2] =")
print(np.array2string(Cp, precision=5))
print("symmetric:", np.allclose(Cp, Cp.T),
      "  positive-definite:", bool(np.all(evals > 0)),
      "  eigs", np.array2string(evals, formatter={'float_kind': lambda z: f'{z:.1e}'}))
print("sigma(px, py, pz) =", np.array2string(np.sqrt(np.diag(Cp)), precision=4), "GeV")

## 4. Differentiable detector design

The first **fit** in the notebook: a differentiable toy detector simulator. The shape
every fit here shares is *a C++ simulator with a loop $\to$ a loss $\to$ Clad gradient
$\to$ fit*.

A sampling calorimeter is a sandwich of dense **absorber** plates (lead, short radiation
length) and thin **active** layers (scintillator, which produces the signal). An electron
of energy $E_0$ showers as it crosses the stack, depositing energy along its depth
following the analytic longitudinal profile $dE/dt \propto (bt)^{a-1} e^{-bt}$ (a Gamma
distribution), with $t$ in radiation lengths. The observable is the **sampling fraction**,
$f_\text{samp} = E_\text{active} / E_\text{total}$. Thicker lead $\to$ more of the shower
dies in passive material $\to$ smaller sampling fraction. The loop over layers is smooth
and elementary (`pow`, `exp`, `log`), so Clad differentiates it directly.

In [ ]:
%%cpp --declare
#include <Math/CladDerivator.h>
#include <cmath>
double sampling_fraction(double d_abs, double E0, int n_layers) {
    const double X0_abs=0.56, X0_act=42.0, d_act=0.5, b=0.5, Ec=0.0074;
    double t_max = std::log(E0/Ec) - 0.5;
    double a = 1.0 + b*t_max;
    double t=0.0, e_act=0.0, e_tot=0.0;
    for (int i=0;i<n_layers;++i){
        double dt_a=d_abs/X0_abs;
        double tc=t+0.5*dt_a; double prof=std::pow(b*tc,a-1.0)*std::exp(-b*tc);
        e_tot+=prof*dt_a; t+=dt_a;
        double dt_b=d_act/X0_act;
        tc=t+0.5*dt_b; prof=std::pow(b*tc,a-1.0)*std::exp(-b*tc);
        e_act+=prof*dt_b; e_tot+=prof*dt_b; t+=dt_b;
    }
    return e_act/e_tot;   // Gamma normalization 1/Gamma(a) cancels here
}
// loss for the inverse problem: (simulated sampling fraction - measured)^2
double loss(double *params, double const *observed) {
    double frac = sampling_fraction(params[0], 10.0, 40);
    double r = frac - observed[0];
    return r*r;
}

In [ ]:
%%cpp
clad::gradient(loss, "params");   // -> loss_grad_0

In [ ]:
print("sampling fraction vs absorber thickness (10 GeV e-, 40 layers):")
for d in [0.2, 0.4, 0.6, 0.8, 1.0, 1.5]:
    print(f"  d_abs = {d:.2f} cm   ->   f_samp = {ROOT.sampling_fraction(d, 10.0, 40):.4%}")

### 4.1 The gradient, and the inverse problem

Pretend a real calorimeter was built with a (secretly known) absorber thickness and we
measured its sampling fraction. One `clad::gradient` gives the exact geometry gradient; a
plain gradient-descent loop then recovers the thickness (with a positivity clamp on the
parameter).

In [ ]:
true_d_abs = 0.60
observed = np.array([ROOT.sampling_fraction(true_d_abs, 10.0, 40)])
print(f"measured sampling fraction: {observed[0]:.4%}  (true d_abs = {true_d_abs} cm, unknown to the fit)")

d_abs = 1.5            # start far from the truth
dparams = np.zeros(1)
learning_rate, n_iter = 3.0e3, 120
for i in range(n_iter):
    params = np.array([d_abs])
    val = ROOT.loss(params, observed)
    dparams[:] = 0.0
    ROOT.loss_grad_0(params, observed, dparams)
    d_abs -= learning_rate * dparams[0]
    if d_abs < 0.05: d_abs = 0.05
    if i % 20 == 0 or i == n_iter-1:
        print(f"iter {i:03d} | loss={val:.3e} | d_abs={d_abs:.4f} cm")
print(f"recovered: {d_abs:.4f} cm   (true {true_d_abs} cm)")

In [ ]:
ds = np.linspace(0.15, 1.6, 100)
fs = [ROOT.sampling_fraction(d, 10.0, 40) for d in ds]
plt.figure(figsize=(6, 4))
plt.plot(ds, fs, label="simulator")
plt.axhline(observed[0], ls="--", color="gray", label="measured")
plt.plot([d_abs], [ROOT.sampling_fraction(d_abs, 10.0, 40)], "o", color="crimson",
         label=f"recovered ({d_abs:.3f} cm)")
plt.xlabel("absorber thickness d_abs [cm]"); plt.ylabel("sampling fraction")
plt.legend(); plt.title("forward model + inverse solution"); plt.show()

### 4.2 The geometry trap: hard vs. soft acceptance

Section 4 so far worked because the observable was smooth. Most raw geometry questions are
NOT — this is the discontinuity trap from section 2, now in a real detector. A particle
hits a module of radius $R$ only if its impact radius $r < R$: a step function, whose
gradient is zero everywhere. The fix (as in section 2) is a smooth **sigmoid** surrogate,
$1/(1 + e^{(r-R)/w})$, which has a real, informative gradient. Impact radii are drawn once
and passed in as data, so the simulator itself has no randomness for Clad to mis-differentiate.

In [ ]:
%%cpp --declare
#include <Math/CladDerivator.h>
#include <cmath>
double acc_hard(double R, double const *r, int n) {           // step function
    double c = 0.0; for (int i=0;i<n;++i) c += (r[i] < R) ? 1.0 : 0.0; return c/n;
}
double acc_soft(double R, double const *r, int n) {           // sigmoid, width w
    const double w = 0.3; double c = 0.0;
    for (int i=0;i<n;++i) c += 1.0/(1.0+std::exp((r[i]-R)/w)); return c/n;
}
double loss_hard(double *p, double const *d){ double a=acc_hard(p[0],d,200); double r=a-0.5; return r*r; }
double loss_soft(double *p, double const *d){ double a=acc_soft(p[0],d,200); double r=a-0.5; return r*r; }

In [ ]:
%%cpp
clad::gradient(loss_hard, "p");
clad::gradient(loss_soft, "p");

In [ ]:
rng = np.random.default_rng(0)
radii = np.sort(rng.uniform(0.0, 10.0, 200))   # fixed impact radii, passed in as data

R = np.array([4.0])
gh = np.zeros(1); ROOT.loss_hard_grad_0(R, radii, gh)
gs = np.zeros(1); ROOT.loss_soft_grad_0(R, radii, gs)
print(f"at R = {R[0]} cm:")
print(f"  hard acceptance = {ROOT.acc_hard(R[0], radii, 200):.3f}   gradient = {gh[0]:+.5f}   <- ZERO: no signal")
print(f"  soft acceptance = {ROOT.acc_soft(R[0], radii, 200):.3f}   gradient = {gs[0]:+.5f}   <- usable")

R_opt = 1.0
for i in range(80):
    pp = np.array([R_opt]); gg = np.zeros(1)
    ROOT.loss_soft_grad_0(pp, radii, gg)
    R_opt -= 20.0 * gg[0]
print(f"optimizing R with the soft gradient -> R = {R_opt:.3f} cm, "
      f"acceptance = {ROOT.acc_soft(R_opt, radii, 200):.3f}  (hard cut cannot be optimized: gradient is 0)")

In [ ]:
Rs = np.linspace(0.0, 10.0, 200)
plt.figure(figsize=(6, 4))
plt.plot(Rs, [ROOT.acc_hard(r, radii, 200) for r in Rs], label="hard (step)")
plt.plot(Rs, [ROOT.acc_soft(r, radii, 200) for r in Rs], label="soft (sigmoid)")
plt.xlabel("module radius R [cm]"); plt.ylabel("acceptance")
plt.legend(); plt.title("differentiable geometry"); plt.show()

## 5. Measuring the Z mass from real CMS open data

The finale: the same recipe as the detector fit, now on **real data**, with a proper
likelihood, a real minimizer (Minuit), and Hessian-based uncertainties. The payoff is an
actual measurement — the Z boson mass, and how many Z bosons are in the sample.

The usual pain of a mass fit is the **normalization** integral (an $\mathrm{erf}$ for a
Gaussian, an $\arctan$ for a Breit-Wigner). A **binned extended-Poisson** fit makes it
vanish: predict a count per bin and normalize by a *sum over bins* instead of an integral,

$$\nu_i = N_s\,\frac{G(m_i)}{\sum_j G(m_j)} + N_b\,\frac{B(m_i)}{\sum_j B(m_j)},$$

with a bare Gaussian $G(m) = e^{-\frac{1}{2}((m-\mu)/\sigma)^2}$ and a falling exponential
background $B(m) = e^{-\lambda m}$, both at bin centers. The extended NLL is
$\sum_i (\nu_i - n_i \ln \nu_i)$. No special functions $\to$ nothing exotic for Clad. The
subtle AD payoff: the normalization $\sum_j G(m_j;\,\mu,\sigma)$ depends on the shape
parameters, so $\partial(\mathrm{NLL})/\partial\mu$ has a term flowing through it — the
piece hand-coded gradients routinely forget, and that Clad gets for free.

The data: 40 bins of 1 GeV over 70-110 GeV of opposite-sign dimuon invariant masses from
the CMS Run2011A DoubleMu dataset (CERN Open Data record 545), pre-binned here so there is
no data loading.

In [ ]:
COUNTS = np.array([
    34, 30, 42, 30, 35, 53, 42, 43, 42, 47, 62, 59, 70, 94, 96, 128,
    171, 263, 422, 675, 806, 873, 595, 314, 228, 107, 80, 54, 46, 31,
    27, 21, 12, 14, 13, 19, 11, 7, 13, 7], dtype=float)
M_LO, M_HI, N_BINS = 70.0, 110.0, 40
BIN_W = (M_HI - M_LO) / N_BINS
CENTERS = M_LO + (np.arange(N_BINS) + 0.5) * BIN_W
PDG_MZ = 91.1876

print(f"{int(COUNTS.sum())} events; tallest bin {CENTERS[COUNTS.argmax()]:.1f} GeV with {int(COUNTS.max())} events")
plt.figure(figsize=(6, 4))
plt.bar(CENTERS, COUNTS, width=BIN_W*0.9, color="steelblue")
plt.xlabel("dimuon invariant mass [GeV]"); plt.ylabel("events / GeV")
plt.title("CMS open data: dimuon spectrum"); plt.show()

### Where the counts come from (reproducing them)

`COUNTS` is not synthetic — it is real CMS data, pre-binned so this notebook needs no
network or file access. The source is the **CMS Run2011A DoubleMu** primary dataset on the
[CERN Open Data Portal, record 545](https://opendata.cern.ch/record/545); the file
`Dimuon_DoubleMu.csv` already carries a precomputed dimuon invariant-mass column `M`
(475k events). The 5716 counts above were produced once with:

```python
import numpy as np

# Download Dimuon_DoubleMu.csv from record 545. Direct location:
#   root://eospublic.cern.ch//eos/opendata/cms/Run2011A/DoubleMu/CSV/12Oct2013-v1/Dimuon_DoubleMu.csv
# (via XRootD; or the https EOS gateway, which needs `curl -k` for its certificate).

d = np.genfromtxt("Dimuon_DoubleMu.csv", delimiter=",", names=True)
opp = d["Q1"] * d["Q2"] < 0                          # opposite-sign muons only
COUNTS, edges = np.histogram(d["M"][opp], bins=40, range=(70, 110))
```

To use a different window or binning, change `range`/`bins` and re-run — everything
downstream (the fit, the Hessian errors) adapts automatically.

### 5.1 The model and likelihood

The parameters `[N_s, N_b, mu, sigma, lambda]` are packed into a `double*` (the same
array-of-parameters shape Clad differentiated in section 1); the bin counts come in as a
`const double*`. From the one NLL we generate both the gradient (for the fit) and the
Hessian (for the error bars).

In [ ]:
%%cpp --declare
#include <Math/CladDerivator.h>
#include <cmath>
const int NBINS = 40; const double MLO = 70.0, MHI = 110.0;
double nll(double *p, double const *n) {
    const double bw = (MHI - MLO) / NBINS;
    double Ns=p[0], Nb=p[1], mu=p[2], sg=p[3], lam=p[4];
    double sumG=0.0, sumB=0.0;
    for (int i=0;i<NBINS;++i){ double m=MLO+(i+0.5)*bw; double z=(m-mu)/sg;
        sumG += std::exp(-0.5*z*z); sumB += std::exp(-lam*m); }
    double val=0.0;
    for (int i=0;i<NBINS;++i){ double m=MLO+(i+0.5)*bw; double z=(m-mu)/sg;
        double G=std::exp(-0.5*z*z), B=std::exp(-lam*m);
        double nu = Ns*G/sumG + Nb*B/sumB;
        val += nu - n[i]*std::log(nu); }
    return val;
}

In [ ]:
%%cpp
clad::gradient(nll, "p");       // -> nll_grad_0
clad::hessian(nll, "p[0:4]");   // -> nll_hessian_0

In [ ]:
def run_fit(nll_fn, grad_fn, start, names, limits):
    npar = len(start)
    def cost(par):
        return nll_fn(np.ascontiguousarray(par, dtype=float), COUNTS)
    def grad(par):
        d = np.zeros(npar); grad_fn(np.ascontiguousarray(par, dtype=float), COUNTS, d); return d
    m = Minuit(cost, start, grad=grad, name=names)
    m.errordef = Minuit.LIKELIHOOD    # plain NLL: 1-sigma at delta-NLL = 0.5
    m.limits = limits
    m.migrad()
    return m
print("run_fit ready")

### 5.2 The fit — Minuit driven by the Clad gradient

No parameter rescaling: the scales are wildly different (yields ~1e3, mu ~ 90,
lambda ~ 0.03), but Minuit carries a per-parameter step size and builds its own metric, so
it fits the **natural** parameters directly. The C++ NLL is the cost; the Clad gradient is
the Jacobian. The extended fit's total prediction matches the observed count.

In [ ]:
m = run_fit(ROOT.nll, ROOT.nll_grad_0,
            start=[4300.0, 1400.0, 91.0, 3.0, 0.03],
            names=["N_s", "N_b", "mu", "sigma", "lambda"],
            limits=[(1, None), (1, None), (80, 100), (0.5, 10), (1e-3, 0.2)])
fit = np.array(m.values); Ns, Nb, mu, sg, lam = fit
print(f"valid minimum: {m.valid}   ({m.nfcn} cost calls, {m.ngrad} gradient calls, NLL = {m.fval:.2f})")
print(f"  N_s = {Ns:8.1f}   N_b = {Nb:8.1f}   (sum {Ns+Nb:.0f}, data {COUNTS.sum():.0f})")
print(f"  mu  = {mu:.4f} GeV   sigma = {sg:.4f} GeV   lambda = {lam:.4f} /GeV")

### 5.3 Uncertainties — invert the Clad Hessian

For a plain NLL the covariance is exactly the inverse Hessian (no factor of two). As an
independent check we compare to Minuit's own numeric Hesse — the two agree, confirming the
Clad Hessian is the exact analytic one.

In [ ]:
H = np.zeros(25); ROOT.nll_hessian_0(np.ascontiguousarray(fit), COUNTS, H)
cov = np.linalg.inv(H.reshape(5, 5)); err = np.sqrt(np.diag(cov))
m.hesse(); minuit_err = np.array(m.errors)

print(f"{'param':6} {'value':>12} {'Clad Hessian':>14} {'Minuit Hesse':>14}")
for name, vv, ee, me in zip(["N_s","N_b","mu","sigma","lambda"], fit, err, minuit_err):
    print(f"{name:6} {vv:12.4f} {ee:14.4f} {me:14.4f}")
print()
print(f"RESULT:  m_Z = {mu:.3f} +/- {err[2]:.3f} GeV (stat.)   PDG: {PDG_MZ:.3f} GeV")
print(f"         N_Z = {Ns:.0f} +/- {err[0]:.0f} reconstructed Z -> mu mu decays")

### 5.4 Is the offset the symmetric line shape? A Breit-Wigner variant

The fitted $\mu$ sits about $0.4$ GeV below the PDG value — far more than the statistical
error. Is that because our Gaussian is symmetric while the Z is a Breit-Wigner with a
low-mass radiative tail? The relativistic BW $S(m) = 1/\big((m^2-M^2)^2 + M^2\Gamma^2\big)$
is a *rational* function, so it stays integral-free and Clad-friendly.

One Clad gotcha worth knowing: the reciprocal must be bound to a **named local**
(`double S = 1.0/den;`) and recomputed each loop; inlining `sumS += 1.0/(...)` makes this
Clad build return a silently wrong gradient. So we always finite-difference-check a fresh
gradient before trusting it.

In [ ]:
%%cpp --declare
#include <Math/CladDerivator.h>
#include <cmath>
double nll_bw(double *p, double const *n) {
    const double bw = (MHI - MLO) / NBINS;
    double Ns=p[0], Nb=p[1], M=p[2], G=p[3], lam=p[4];
    double sumS=0.0, sumB=0.0;
    for (int i=0;i<NBINS;++i){ double m=MLO+(i+0.5)*bw;
        double d=m*m-M*M; double den=d*d+M*M*G*G; double S=1.0/den; sumS+=S;
        double B=std::exp(-lam*m); sumB+=B; }
    double val=0.0;
    for (int i=0;i<NBINS;++i){ double m=MLO+(i+0.5)*bw;
        double d=m*m-M*M; double den=d*d+M*M*G*G; double S=1.0/den;
        double B=std::exp(-lam*m);
        double nu=Ns*S/sumS+Nb*B/sumB; val += nu - n[i]*std::log(nu); }
    return val;
}

In [ ]:
%%cpp
clad::gradient(nll_bw, "p");
clad::hessian(nll_bw, "p[0:4]");

In [ ]:
# finite-difference check of the fresh gradient (see the gotcha above)
p_chk = np.array([4300.0, 1400.0, 91.0, 3.0, 0.03])
ga = np.zeros(5); ROOT.nll_bw_grad_0(np.ascontiguousarray(p_chk), COUNTS, ga)
gn = np.zeros(5)
for i in range(5):
    h = 1e-6 * max(1.0, abs(p_chk[i]))
    pp = p_chk.copy(); pp[i]+=h; pm = p_chk.copy(); pm[i]-=h
    gn[i] = (ROOT.nll_bw(np.ascontiguousarray(pp), COUNTS) -
             ROOT.nll_bw(np.ascontiguousarray(pm), COUNTS)) / (2*h)
print("gradient check vs finite differences:", "PASS" if np.allclose(ga, gn, rtol=1e-3) else "FAIL")

m_bw = run_fit(ROOT.nll_bw, ROOT.nll_bw_grad_0,
               start=[4300.0, 1400.0, 91.0, 3.0, 0.03],
               names=["N_s","N_b","M","Gamma","lambda"],
               limits=[(1,None),(1,None),(85,97),(0.5,12),(1e-3,0.2)])
fit_bw = np.array(m_bw.values); Ns_bw, Nb_bw, M_bw, G_bw, lam_bw = fit_bw
print(f"Breit-Wigner fit:  m_Z = {M_bw:.3f} GeV,  Gamma = {G_bw:.3f} GeV   (NLL {m_bw.fval:.1f})")
print(f"Gaussian fit    :  m_Z = {mu:.3f} GeV,  sigma = {sg:.3f} GeV   (NLL {m.fval:.1f})")
print("(the +/- uncertainties come from the exercise below)")

**Exercise.** You just fit the Breit-Wigner and generated `nll_bw_hessian_0`. Get its **uncertainty**
the same way as section 5.3: fill the 5×5 Clad Hessian at the minimum, invert it for the
covariance, and read the 1-sigma error on $m_Z$ (parameter index 2). Store it as `err_bw`
— the fit plot in 5.5 uses it.

In [ ]:
# EXERCISE: get the Breit-Wigner covariance from the Clad Hessian.
#   1. Hb = np.zeros(25); ROOT.nll_bw_hessian_0(np.ascontiguousarray(fit_bw), COUNTS, Hb)
#   2. cov = np.linalg.inv(Hb.reshape(5, 5))
#   3. err_bw = np.sqrt(np.diag(cov))
#   4. print m_Z = M_bw +/- err_bw[2]
# (Define err_bw; the section 5.5 plot depends on it.)


**Solution.**

In [ ]:
Hb = np.zeros(25)
ROOT.nll_bw_hessian_0(np.ascontiguousarray(fit_bw), COUNTS, Hb)
err_bw = np.sqrt(np.diag(np.linalg.inv(Hb.reshape(5, 5))))
print(f"Breit-Wigner:  m_Z = {M_bw:.3f} +/- {err_bw[2]:.3f} GeV,  Gamma = {G_bw:.3f} +/- {err_bw[3]:.3f} GeV")
print(f"Gaussian    :  m_Z = {mu:.3f} +/- {err[2]:.3f} GeV,  sigma = {sg:.3f} +/- {err[3]:.3f} GeV")
print(f"PDG         :  m_Z = {PDG_MZ:.3f} GeV,  Gamma = 2.495 GeV")


**The symmetric line shape was not the whole story.** The Breit-Wigner fits better
(lower NLL, a physical width) but $m_Z$ only creeps up by $\sim 0.07$ GeV — still
$\sim 0.4$ GeV below PDG, far more than the $\sim 0.04$ GeV statistical error. That residual
is a **systematic**, dominated by the neglected QED final-state-radiation (radiative-return)
tail, which pulls the reconstructed dimuon mass low and which a *symmetric* line shape
cannot capture — plus residual muon momentum-scale calibration. The real fix is an
asymmetric tail (a Crystal Ball), not a better symmetric peak. The lesson stands: with
plentiful statistics you fight systematics, not statistics — Clad gave the $\sim 0.04$ GeV
statistical error for free; the $\sim 0.4$ GeV that dominates is physics.

### 5.5 The fit

In [ ]:
def gauss_bkg(par):
    Ns, Nb, mu, sg, lam = par
    G = np.exp(-0.5*((CENTERS-mu)/sg)**2); B = np.exp(-lam*CENTERS)
    return Ns*G/G.sum(), Nb*B/B.sum()
def bw_bkg(par):
    Ns, Nb, M, G, lam = par
    d = CENTERS**2 - M**2; S = 1.0/(d*d + M*M*G*G); B = np.exp(-lam*CENTERS)
    return Ns*S/S.sum(), Nb*B/B.sum()

sig_g, bkg_g = gauss_bkg(fit); sig_b, bkg_b = bw_bkg(fit_bw)
plt.figure(figsize=(7.5, 5))
plt.errorbar(CENTERS, COUNTS, yerr=np.sqrt(COUNTS), fmt="o", ms=4, color="black", label="CMS open data", zorder=5)
plt.plot(CENTERS, sig_b+bkg_b, "-", color="crimson", lw=2, label=f"Breit-Wigner (m_Z={M_bw:.2f})")
plt.plot(CENTERS, sig_g+bkg_g, "--", color="seagreen", lw=1.8, label=f"Gaussian (m_Z={mu:.2f})")
plt.plot(CENTERS, bkg_b, ":", color="steelblue", label="background (BW)")
plt.axvline(PDG_MZ, color="gray", ls=":", alpha=0.8)
plt.xlabel("dimuon invariant mass [GeV]"); plt.ylabel("events / GeV")
plt.title(f"Z peak from CMS open data: m_Z = {M_bw:.2f} +/- {err_bw[2]:.2f} (stat) +/- {PDG_MZ-M_bw:.1f} (syst) GeV")
plt.legend(); plt.show()

## Conclusions

Across five sections the same tool did a lot of different work, all from ordinary C++:

- **exact gradients** for optimization (`clad::gradient`), forward and reverse mode and
  when each wins (`clad::differentiate` vs `clad::gradient`);
- **Jacobians** and the **jvp / vjp** duality that carries straight over to JAX
  (`clad::jacobian` is vectorized forward mode = `jacfwd`);
- **Hessians** for uncertainties ($C = H^{-1}$), cross-checked against Minuit;
- and the essential caveat: **AD differentiates the code you wrote, not the math you
  meant** — discontinuities silently corrupt gradients, so keep models smooth and
  finite-difference-check.

We ended with a real measurement — the Z mass and yield from CMS open data — where the
punchline was not the AD but the physics: once statistics are plentiful, **systematics**
dominate.

### The tip of the iceberg

These toy examples barely scratch the surface of AD in the physical sciences. A few of the
directions they point to:

- **Particle physics.** Gradient-based *detector design and optimization* (differentiable
  simulators, smooth surrogates for hard cuts — our section 4 in miniature); differentiable
  matrix elements and event generators (e.g. MadJax); systematic-aware *analysis
  optimization* (INFERNO, neos); gradients through track and vertex fits (the Jacobians and
  $C' = J\,C\,J^\top$ of section 3); auto-tuning of simulation and reconstruction parameters.
- **Nuclear physics.** Gradient-based fitting of nuclear energy-density functionals and
  optical-model potentials; differentiable transport and inverse problems in imaging;
  optimal experimental design.
- **Astrophysics and cosmology.** Differentiable cosmology pipelines (jax-cosmo) and
  gravitational-wave analysis, where the Fisher matrix $F = J^\top C^{-1} J$ — exactly the
  covariance-propagation machinery of section 3 — drives parameter forecasts; differentiable
  N-body, ray-tracing and lensing; gradient-based sampling (HMC/NUTS) and simulation-based
  inference, all of which need derivatives of the model.

The common thread is Clad's promise: you do not rewrite your physics into a tensor
framework to get gradients. **If you can write it in C++, you can differentiate it** — and
plug the result straight into an optimizer, a sampler, or an error propagation.